In [2]:
import pandas as pd
import numpy as np
import joblib

model = joblib.load("../models/ridge_pm25_forecasting.joblib")

print("Model berhasil di-load.")
print(model)

Model berhasil di-load.
Pipeline(steps=[('scaler', StandardScaler()), ('ridge', Ridge())])


In [3]:
df_new = pd.read_csv(
    "../data/raw/merged_1_datetime.csv",
    parse_dates=["DATETIME"]
)

print("Shape:", df_new.shape)
print("Periode:", df_new["DATETIME"].min(), "→", df_new["DATETIME"].max())
print("\nKolom:")
print(df_new.columns.tolist())

Shape: (324988, 10)
Periode: 2026-07-01 00:00:01 → 2026-09-21 17:40:02

Kolom:
['NO2', 'SO2', 'PM25', 'PM10', 'DATETIME', 'TMP', 'RAIN', 'HUM', 'WS', 'WD']


In [4]:
df_new["DATETIME_5MIN"] = df_new["DATETIME"].dt.floor("5min")

df_new_5min = (
    df_new
    .sort_values("DATETIME")
    .copy()
)

print("Jumlah baris setelah normalisasi timestamp:",
      df_new_5min["DATETIME_5MIN"].nunique())

print(
    "Periode:",
    df_new_5min["DATETIME_5MIN"].min(),
    "→",
    df_new_5min["DATETIME_5MIN"].max()
)

Jumlah baris setelah normalisasi timestamp: 23533
Periode: 2026-07-01 00:00:00 → 2026-09-21 17:40:00


In [5]:
df_new_5min = (
    df_new
    .groupby("DATETIME_5MIN")
    .agg({
        "NO2": "mean",
        "SO2": "mean",
        "PM25": "mean",
        "PM10": "mean",
        "TMP": "mean",
        "HUM": "mean",
        "WS": "mean"
    })
    .reset_index()
)

print("Shape:", df_new_5min.shape)
print("\nMissing values:")
print(df_new_5min.isna().sum())

Shape: (23533, 8)

Missing values:
DATETIME_5MIN    0
NO2              0
SO2              0
PM25             0
PM10             0
TMP              0
HUM              0
WS               0
dtype: int64


In [6]:
df_new_hourly = (
    df_new_5min
    .set_index("DATETIME_5MIN")
    .resample("1h")
    .mean()
    .reset_index()
)

print("Shape:", df_new_hourly.shape)

print("\nPeriode:")
print(
    df_new_hourly["DATETIME_5MIN"].min(),
    "→",
    df_new_hourly["DATETIME_5MIN"].max()
)

print("\nMissing values:")
print(df_new_hourly.isna().sum())

Shape: (1986, 8)

Periode:
2026-07-01 00:00:00 → 2026-09-21 17:00:00

Missing values:
DATETIME_5MIN    0
NO2              3
SO2              3
PM25             3
PM10             3
TMP              3
HUM              3
WS               3
dtype: int64


In [7]:
df_new["WD_sin"] = np.sin(np.deg2rad(df_new["WD"]))
df_new["WD_cos"] = np.cos(np.deg2rad(df_new["WD"]))

wd_new_hourly = (
    df_new
    .set_index("DATETIME")
    .resample("1h")[["WD_sin", "WD_cos"]]
    .mean()
    .reset_index()
)

print("Shape WD hourly:", wd_new_hourly.shape)
print("\nMissing values:")
print(wd_new_hourly.isna().sum())

Shape WD hourly: (1986, 3)

Missing values:
DATETIME    0
WD_sin      3
WD_cos      3
dtype: int64


In [ ]:
df_new_model = df_new_hourly.merge(
    wd_new_hourly,
    left_on="DATETIME_5MIN",
    right_on="DATETIME",
    how="left"
)

df_new_model = df_new_model.drop(columns=["DATETIME"])

print("Shape:", df_new_model.shape)

print("\nKolom:")
print(df_new_model.columns.tolist())

print("\nMissing values:")
print(df_new_model.isna().sum())

Shape: (1986, 10)

Kolom:
['DATETIME_5MIN', 'NO2', 'SO2', 'PM25', 'PM10', 'TMP', 'HUM', 'WS', 'WD_sin', 'WD_cos']

Missing values:
DATETIME_5MIN    0
NO2              3
SO2              3
PM25             3
PM10             3
TMP              3
HUM              3
WS               3
WD_sin           3
WD_cos           3
dtype: int64


In [9]:
for lag in [1, 2, 3, 6, 24]:
    df_new_model[f"PM25_lag_{lag}"] = (
        df_new_model["PM25"].shift(lag)
    )

print(df_new_model[
    [
        "DATETIME_5MIN",
        "PM25",
        "PM25_lag_1",
        "PM25_lag_2",
        "PM25_lag_3",
        "PM25_lag_6",
        "PM25_lag_24"
    ]
].tail())

           DATETIME_5MIN        PM25  PM25_lag_1  PM25_lag_2  PM25_lag_3  \
1981 2026-09-21 13:00:00  128.356494  123.506505  123.817590   93.567515   
1982 2026-09-21 14:00:00  123.154360  128.356494  123.506505  123.817590   
1983 2026-09-21 15:00:00   95.276561  123.154360  128.356494  123.506505   
1984 2026-09-21 16:00:00   76.864691   95.276561  123.154360  128.356494   
1985 2026-09-21 17:00:00   55.423392   76.864691   95.276561  123.154360   

      PM25_lag_6  PM25_lag_24  
1981  137.799823    64.454076  
1982   57.522035    97.554054  
1983   60.651198    82.235707  
1984   93.567515    91.146989  
1985  123.817590    47.586026  


In [10]:
df_new_model["PM25_roll_mean_3"] = (
    df_new_model["PM25"].rolling(window=3).mean()
)

df_new_model["PM25_roll_mean_6"] = (
    df_new_model["PM25"].rolling(window=6).mean()
)

print(df_new_model[
    [
        "DATETIME_5MIN",
        "PM25",
        "PM25_roll_mean_3",
        "PM25_roll_mean_6"
    ]
].tail())

           DATETIME_5MIN        PM25  PM25_roll_mean_3  PM25_roll_mean_6
1981 2026-09-21 13:00:00  128.356494        125.226863         97.903556
1982 2026-09-21 14:00:00  123.154360        125.005787        108.842277
1983 2026-09-21 15:00:00   95.276561        115.595805        114.613171
1984 2026-09-21 16:00:00   76.864691         98.431871        111.829367
1985 2026-09-21 17:00:00   55.423392         75.854881        100.430334


In [11]:
df_new_model["hour"] = df_new_model["DATETIME_5MIN"].dt.hour
df_new_model["day_of_week"] = df_new_model["DATETIME_5MIN"].dt.dayofweek

df_new_model["hour_sin"] = np.sin(
    2 * np.pi * df_new_model["hour"] / 24
)

df_new_model["hour_cos"] = np.cos(
    2 * np.pi * df_new_model["hour"] / 24
)

print(df_new_model[
    [
        "DATETIME_5MIN",
        "hour",
        "day_of_week",
        "hour_sin",
        "hour_cos"
    ]
].tail())

           DATETIME_5MIN  hour  day_of_week  hour_sin  hour_cos
1981 2026-09-21 13:00:00    13            0 -0.258819 -0.965926
1982 2026-09-21 14:00:00    14            0 -0.500000 -0.866025
1983 2026-09-21 15:00:00    15            0 -0.707107 -0.707107
1984 2026-09-21 16:00:00    16            0 -0.866025 -0.500000
1985 2026-09-21 17:00:00    17            0 -0.965926 -0.258819


In [12]:
feature_cols = [
    "PM25_lag_1",
    "PM25_lag_2",
    "PM25_lag_3",
    "PM25_lag_6",
    "PM25_lag_24",
    "PM25_roll_mean_3",
    "PM25_roll_mean_6",
    "PM10",
    "NO2",
    "SO2",
    "TMP",
    "HUM",
    "WS",
    "WD_sin",
    "WD_cos",
    "hour",
    "day_of_week",
    "hour_sin",
    "hour_cos"
]

X_new = df_new_model[feature_cols].iloc[[-1]]

print("Shape X_new:", X_new.shape)
print("\nFitur yang digunakan:")
print(X_new.columns.tolist())

print("\nData untuk prediksi:")
print(X_new)

Shape X_new: (1, 19)

Fitur yang digunakan:
['PM25_lag_1', 'PM25_lag_2', 'PM25_lag_3', 'PM25_lag_6', 'PM25_lag_24', 'PM25_roll_mean_3', 'PM25_roll_mean_6', 'PM10', 'NO2', 'SO2', 'TMP', 'HUM', 'WS', 'WD_sin', 'WD_cos', 'hour', 'day_of_week', 'hour_sin', 'hour_cos']

Data untuk prediksi:
      PM25_lag_1  PM25_lag_2  PM25_lag_3  PM25_lag_6  PM25_lag_24  \
1985   76.864691   95.276561   123.15436   123.81759    47.586026   

      PM25_roll_mean_3  PM25_roll_mean_6       PM10        NO2       SO2  \
1985         75.854881        100.430334  99.176302  11.193537  1.771957   

            TMP        HUM        WS    WD_sin   WD_cos  hour  day_of_week  \
1985  34.011652  66.491052  1.391176  0.361967 -0.13422    17            0   

      hour_sin  hour_cos  
1985 -0.965926 -0.258819  


In [13]:
pred_pm25 = model.predict(X_new)[0]

print(f"Prediksi PM2.5 untuk 1 jam berikutnya: {pred_pm25:.2f} µg/m³")

Prediksi PM2.5 untuk 1 jam berikutnya: 52.96 µg/m³
